# Testing


In [2]:
import pandas as pd
import numpy
import matplotlib
import seaborn
import sqlalchemy
import yfinance as yf
from fredapi import Fred

---
## Macroeconomic Data (Fred API)

In [3]:
fred = Fred(api_key="7f0fb7b341f36af702bff6dba49512d3")

# tables of macroeconomic data from Fred
cpi = fred.get_series("cpiaucsl")
interest = fred.get_series("fedfunds")
unemployment = fred.get_series("unrate")
market_expect = fred.get_series("dgs10")
volatility = fred.get_series("vixcls")

### EDA of CPI

In [4]:
cpi.isnull().sum()
cpi_df = pd.DataFrame(cpi, columns=["cpi"])
cpi_df.isnull().sum()
cpi_df.head()
cpi_df.shape
cpi_df

#monthly data

,cpi
1947-01-01,21.480
1947-02-01,21.620
1947-03-01,22.000
1947-04-01,22.000
1947-05-01,21.950
...,...
2025-11-01,325.063
2025-12-01,326.031
2026-01-01,326.588
2026-02-01,327.460


### EDA of interest

In [5]:
interest_df = pd.DataFrame(interest, columns=["fedfunds_rate"])
print(interest_df.isnull().sum())
interest_df.head()
interest_df.shape
interest_df

#monthly data

fedfunds_rate    0
dtype: int64


,fedfunds_rate
1954-07-01,0.80
1954-08-01,1.22
1954-09-01,1.07
1954-10-01,0.85
1954-11-01,0.83
...,...
2025-11-01,3.88
2025-12-01,3.72
2026-01-01,3.64
2026-02-01,3.64


### EDA of unemployment

In [6]:
unemployment_df = pd.DataFrame(unemployment, columns=["unemployment_rate"])
print(unemployment_df.isnull().sum())
unemployment_df.head()
unemployment_df.shape
unemployment_df

#monthly data

unemployment_rate    1
dtype: int64


,unemployment_rate
1948-01-01,3.4
1948-02-01,3.8
1948-03-01,4.0
1948-04-01,3.9
1948-05-01,3.5
...,...
2025-11-01,4.5
2025-12-01,4.4
2026-01-01,4.3
2026-02-01,4.4


### EDA of market_expect

In [7]:
market_expect_df = pd.DataFrame(market_expect, columns=["market_expect"])
market_expect_df.isnull().sum()
market_expect_df.head()
market_expect_df.shape
market_expect_df



#daily data

,market_expect
1962-01-02,4.06
1962-01-03,4.03
1962-01-04,3.99
1962-01-05,4.02
1962-01-08,4.03
...,...
2026-04-10,4.31
2026-04-13,4.30
2026-04-14,4.26
2026-04-15,4.29


### EDA of volatility

In [8]:
volatility_df = pd.DataFrame(volatility, columns=["volatility"])
volatility_df.isnull().sum()
volatility_df.head()
volatility_df.shape
volatility_df

#daily data

,volatility
1990-01-02,17.24
1990-01-03,18.19
1990-01-04,19.22
1990-01-05,20.11
1990-01-08,20.26
...,...
2026-04-13,19.12
2026-04-14,18.36
2026-04-15,18.17
2026-04-16,17.94


Note: decide if the market expectation and volatility is really needed

### Inspection of CPI and unemployment null values

In [9]:
print(cpi_df[cpi_df["cpi"].isnull()])
print(unemployment_df[unemployment_df["unemployment_rate"].isnull()])

# the null for both of these are the same date 2025-10-01


            cpi
2025-10-01  NaN
            unemployment_rate
2025-10-01                NaN


### Creating table for macroeconomic values

In [65]:
macro = cpi_df.join([interest_df, unemployment_df], how="outer")
macro.dropna()
macro_filtered = macro.loc["1954-07-01":"2025-09-01"]
macro_filtered.isnull().sum()
macro_filtered = macro_filtered.reset_index()
macro_df = macro_filtered.rename(columns={"index":"date"})
macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_df

,date,cpi,fedfunds_rate,unemployment_rate
0,1954-07-01,26.860,0.80,5.8
1,1954-08-01,26.850,1.22,6.0
2,1954-09-01,26.810,1.07,6.1
3,1954-10-01,26.720,0.85,5.7
4,1954-11-01,26.780,0.83,5.3
...,...,...,...,...
850,2025-05-01,320.620,4.33,4.3
851,2025-06-01,321.435,4.33,4.1
852,2025-07-01,322.169,4.33,4.3
853,2025-08-01,323.291,4.33,4.3


## Stock data (yfinance)

In [ ]:
stock = yf.download("^GSPC", period="max")
stock_df = pd.DataFrame(stock)
stock_close = stock_df[["Close"]]
stock_close.columns = stock_close.columns.get_level_values(0)
stock_close

#daily data

[*********************100%***********************]  1 of 1 completed


Price,Close
Date,
1927-12-30,17.660000
1928-01-03,17.760000
1928-01-04,17.719999
1928-01-05,17.549999
1928-01-06,17.660000
...,...
2026-04-14,6967.379883
2026-04-15,7022.950195
2026-04-16,7041.279785


### turn daily stock data into monthly

In [46]:
stock_close_monthly = stock_close.resample('MS').last()
stock_close_monthly
stock_close_monthly_filtered = stock_close_monthly.loc["1954-07-01":"2025-09-01"]
stock_close_monthly_filtered.shape
stock_close_monthly_filtered

Price,Close
Date,
1954-07-01,30.879999
1954-08-01,29.830000
1954-09-01,32.310001
1954-10-01,31.680000
1954-11-01,34.240002
...,...
2025-05-01,5911.689941
2025-06-01,6204.950195
2025-07-01,6339.390137


In [50]:
sp500 = stock_close_monthly_filtered.rename(columns={"Close": "sp500"})
sp500 = sp500.reset_index()
sp500 = sp500.rename(columns={"Date": "date"})
sp500["date"] = pd.to_datetime(sp500["date"])
sp500

Price,date,sp500
0,1954-07-01,30.879999
1,1954-08-01,29.830000
2,1954-09-01,32.310001
3,1954-10-01,31.680000
4,1954-11-01,34.240002
...,...,...
850,2025-05-01,5911.689941
851,2025-06-01,6204.950195
852,2025-07-01,6339.390137
853,2025-08-01,6460.259766


## merging macro table with stock table

In [68]:
stock_macro = pd.merge(sp500, macro_df, on="date", how="outer")
stock_macro

stock_macro.isnull().sum()

date                 0
sp500                0
cpi                  0
fedfunds_rate        0
unemployment_rate    0
dtype: int64